# myPolar – Fine-tuning Pre-trained Models (XLM-RoBERTa, mBERT)

Same data and tasks as `myPolar_ML.ipynb` and `myPolar_DL.ipynb`.

A **pre-trained model** has already read huge amounts of text in about 100 languages, including Burmese.
We do not train it from zero: we add one small layer on top and **fine-tune** it on our data for a few epochs.

| Model | Hugging Face name |
|---|---|
| XLM-RoBERTa (base) | `FacebookAI/xlm-roberta-base` |
| Multilingual BERT | `google-bert/bert-base-multilingual-cased` |

**Plan:** train on *train* → keep the best epoch on *dev* → choose thresholds on *dev* → score once on *test*.

### Kaggle settings (right-hand panel, before running)
1. **Accelerator → GPU T4 x2** (or P100)
2. **Internet → On** (needed to download the models)
3. **Add Input → your dataset** with the 9 csv files from `myPolar/data`

## Step 1. Settings, GPU check and data

In [4]:
import os, glob, random, copy, warnings
warnings.filterwarnings("ignore")                 # keep the output clean

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, classification_report
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

# ---------- settings you may change ----------
PRETRAINED_MODELS = [
    "FacebookAI/xlm-roberta-base",
    "google-bert/bert-base-multilingual-cased",   # remove this line to save time
]
MAX_LEN    = 128        # tokens per sentence (longer sentences are cut)
BATCH_SIZE = 16
EPOCHS     = 4          # pre-trained models need only a few epochs
LR         = 2e-5       # small learning rate: we only adjust the model a little
SEED       = 42

def set_seed(seed=SEED):
    """Same seed -> same random numbers -> same results every run."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "no GPU -> turn on GPU in Kaggle settings!")

# ---------- find the data folder automatically ----------
# On Kaggle the dataset is somewhere under /kaggle/input/...; on a laptop it is the local "data" folder.
found = glob.glob("/kaggle/input/datasets/ohnmarnyuntkic/mypolar/task1_polarization_train.csv", recursive=True)
DATA_DIR = os.path.dirname(found[0]) if found else "data"
print("Data folder:", DATA_DIR)

TASKS = {
    "task1": ("task1_polarization",  ["polarization"]),
    "task2": ("task2_type",          ["political", "racial_ethnic", "religious", "gender_sexual", "other"]),
    "task3": ("task3_manifestation", ["stereotype", "vilification", "dehumanization",
                                      "extreme_language", "lack_of_empathy", "invalidation"]),
}
data = {}
for task, (prefix, labels) in TASKS.items():
    data[task] = {split: pd.read_csv(f"{DATA_DIR}/{prefix}_{split}.csv", keep_default_na=False)
                  for split in ["train", "dev", "test"]}
    print(f"{task}: train={len(data[task]['train'])}  dev={len(data[task]['dev'])}  "
          f"test={len(data[task]['test'])}  labels={len(labels)}")

Device: cuda | Tesla T4
Data folder: /kaggle/input/datasets/ohnmarnyuntkic/mypolar
task1: train=7668  dev=1643  test=1644  labels=1
task2: train=3570  dev=766  test=771  labels=5
task3: train=3324  dev=710  test=709  labels=6


## Step 2. Tokenizer: text → token IDs
Each pre-trained model has its **own tokenizer**, so we use the raw `text` column (not our syllables).
Let's see how each model cuts one Burmese sentence and how long the sentences become.

In [5]:
example = data["task1"]["train"]["text"][0]
print("Sentence:", example[:60], "\n")

for name in PRETRAINED_MODELS:
    tok = AutoTokenizer.from_pretrained(name)
    lengths = [len(tok.tokenize(s)) for s in data["task1"]["train"]["text"][:500]]
    print(f"{name}")
    print("   tokens:", tok.tokenize(example)[:10])
    print(f"   sentence length in tokens: median {int(np.median(lengths))}, "
          f"{np.mean(np.array(lengths) <= MAX_LEN):.0%} fit in MAX_LEN={MAX_LEN}\n")

Sentence: မဟာသရေစည်သူဦးသန့်၂၂ဇန်နဝါရီ၁၉၀၉ – ၂၅နိုဝင်ဘာ၁၉၇၄သည်မြန်မာနို 



config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

FacebookAI/xlm-roberta-base
   tokens: ['▁မ', 'ဟာ', 'သ', 'ရ', 'ေစ', 'ည်', 'သူ', 'ဦး', 'သ', 'န့်']
   sentence length in tokens: median 30, 99% fit in MAX_LEN=128



config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

google-bert/bert-base-multilingual-cased
   tokens: ['မ', '##ဟ', '##ာ', '##သ', '##ရ', '##ေ', '##စ', '##ည်', '##သ', '##ူ']
   sentence length in tokens: median 39, 98% fit in MAX_LEN=128



## Step 3. DataLoader: give the data to the model in small batches
The tokenizer returns `input_ids` (token numbers) and `attention_mask` (1 = real token, 0 = padding).

In [6]:
def make_loader(task, split, tokenizer, shuffle=False):
    df = data[task][split]
    labels = TASKS[task][1]
    enc = tokenizer(list(df["text"]), padding="max_length", truncation=True,
                    max_length=MAX_LEN, return_tensors="pt")
    y = torch.tensor(df[labels].values, dtype=torch.float)          # labels: 0 / 1
    dataset = TensorDataset(enc["input_ids"], enc["attention_mask"], y)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle)

tok = AutoTokenizer.from_pretrained(PRETRAINED_MODELS[0])
ids, mask, y = next(iter(make_loader("task2", "train", tok)))
print("One batch -> input_ids:", tuple(ids.shape), " attention_mask:", tuple(mask.shape), " y:", tuple(y.shape))

One batch -> input_ids: (16, 128)  attention_mask: (16, 128)  y: (16, 5)


## Step 4. The model: pre-trained encoder + one new layer
- The **encoder** turns the sentence into vectors (this part is pre-trained).
- We take the vector of the **first token** (it summarises the whole sentence).
- A new **Linear layer** gives one score per label (Task 1: 1, Task 2: 5, Task 3: 6).

In [7]:
class PretrainedClassifier(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)                     # pre-trained part
        self.dropout = nn.Dropout(0.1)
        self.out = nn.Linear(self.encoder.config.hidden_size, num_labels)        # our new layer

    def forward(self, input_ids, attention_mask):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        return self.out(self.dropout(h[:, 0]))                                   # first token -> scores

m = PretrainedClassifier(PRETRAINED_MODELS[0], num_labels=1)
print(f"{PRETRAINED_MODELS[0]} parameters: {sum(p.numel() for p in m.parameters()):,}")
del m

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FacebookAI/xlm-roberta-base parameters: 278,044,417


## Step 5. Helper functions: score, thresholds, training, testing
- **Score** = macro-F1 (same as the other notebooks).
- **Loss** = `BCEWithLogitsLoss` with `pos_weight`: mistakes on rare labels cost more.
- **Learning-rate schedule**: start gently (warm-up), then slowly lower the learning rate.
- **Best epoch**: after every epoch we check dev and keep the best one.
- **Thresholds**: we try cut-offs other than 0.5 on **dev** and keep the best one for each label (never on test).

In [8]:
def macro_f1(task, y_true, y_pred):
    # Task 1: average F1 of class 0 and class 1.  Task 2/3: average F1 over the labels.
    if task == "task1":
        return f1_score(y_true.ravel(), y_pred.ravel(), average="macro", zero_division=0)
    return f1_score(y_true, y_pred, average="macro", zero_division=0)


def get_probs(model, loader):
    """Run the model without training and return probabilities and true labels."""
    model.eval()
    all_probs, all_y = [], []
    with torch.no_grad():
        for ids, mask, y in loader:
            logits = model(ids.to(device), mask.to(device))
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_y.append(y.numpy())
    return np.vstack(all_probs), np.vstack(all_y).astype(int)


def tune_thresholds(task, probs, y):
    """For each label, try cut-offs 0.05 ... 0.95 on dev and keep the best one."""
    grid = np.arange(0.05, 0.96, 0.05)
    best = []
    for j in range(y.shape[1]):
        scores = [macro_f1(task, y[:, [j]], (probs[:, [j]] >= t).astype(int)) if task == "task1"
                  else f1_score(y[:, j], (probs[:, j] >= t).astype(int), zero_division=0)
                  for t in grid]
        best.append(grid[int(np.argmax(scores))])
    return np.array(best)


def train_model(model_name, task, tokenizer):
    set_seed()
    model = PretrainedClassifier(model_name, num_labels=len(TASKS[task][1])).to(device)
    train_loader = make_loader(task, "train", tokenizer, shuffle=True)
    dev_loader   = make_loader(task, "dev", tokenizer)

    # pos_weight = (#negatives / #positives) for every label -> rare labels get a bigger weight
    y_train = data[task]["train"][TASKS[task][1]].values
    pos_weight = torch.tensor((len(y_train) - y_train.sum(0)) / y_train.sum(0).clip(min=1),
                              dtype=torch.float).to(device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps),
                                                num_training_steps=total_steps)

    best_f1, best_state = -1, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0
        for ids, mask, y in train_loader:
            optimizer.zero_grad()                                    # 1) clear old gradients
            logits = model(ids.to(device), mask.to(device))          # 2) predict
            loss = loss_fn(logits, y.to(device))                     # 3) how wrong?
            loss.backward()                                          # 4) compute gradients
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)        #    (avoid too big steps)
            optimizer.step()                                         # 5) update the weights
            scheduler.step()                                         # 6) adjust the learning rate
            total_loss += loss.item()

        probs, y_dev = get_probs(model, dev_loader)
        dev_f1 = macro_f1(task, y_dev, (probs >= 0.5).astype(int))
        mark = ""
        if dev_f1 > best_f1:                                         # remember the best epoch
            best_f1, mark = dev_f1, "  <- best"
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"epoch {epoch} | train loss {total_loss / len(train_loader):.4f} | dev macro-F1 {dev_f1:.4f}{mark}")

    model.load_state_dict(best_state)                                # go back to the best epoch
    return model


def evaluate(model, task, model_name, tokenizer):
    labels = TASKS[task][1]
    dev_probs,  y_dev  = get_probs(model, make_loader(task, "dev",  tokenizer))
    test_probs, y_test = get_probs(model, make_loader(task, "test", tokenizer))

    thresholds = tune_thresholds(task, dev_probs, y_dev)              # chosen on dev only
    dev_f1  = macro_f1(task, y_dev,  (dev_probs  >= thresholds).astype(int))
    y_pred  = (test_probs >= thresholds).astype(int)
    test_f1 = macro_f1(task, y_test, y_pred)

    print(f"\n>>> {model_name} | {task} | thresholds {np.round(thresholds, 2).tolist()}")
    print(f">>> dev macro-F1 {dev_f1:.4f} | TEST macro-F1 {test_f1:.4f}")
    if task == "task1":
        print(classification_report(y_test.ravel(), y_pred.ravel(),
                                    target_names=["non-polarized", "polarized"], digits=3, zero_division=0))
    else:
        print(classification_report(y_test, y_pred, target_names=labels, digits=3, zero_division=0))
    return {"task": task, "model": model_name.split("/")[-1],
            "dev_macro_f1": round(dev_f1, 4), "test_macro_f1": round(test_f1, 4)}


results = []      # every final score is stored here

## Step 6. Fine-tune every model on every task
One model per task. On a Kaggle T4 GPU this takes roughly 20–30 minutes per pre-trained model.

In [9]:
for model_name in PRETRAINED_MODELS:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    for task in TASKS:
        print(f"\n========== {model_name} | {task} ==========")
        model = train_model(model_name, task, tokenizer)
        results.append(evaluate(model, task, model_name, tokenizer))
        del model
        torch.cuda.empty_cache()                                     # free GPU memory for the next model


========== FacebookAI/xlm-roberta-base | task1 ==========


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 1 | train loss 0.6195 | dev macro-F1 0.7419  <- best
epoch 2 | train loss 0.4883 | dev macro-F1 0.7758  <- best
epoch 3 | train loss 0.4149 | dev macro-F1 0.7792  <- best
epoch 4 | train loss 0.3423 | dev macro-F1 0.7973  <- best

>>> FacebookAI/xlm-roberta-base | task1 | thresholds [0.55]
>>> dev macro-F1 0.8002 | TEST macro-F1 0.8016
               precision    recall  f1-score   support

non-polarized      0.792     0.821     0.806       827
    polarized      0.812     0.782     0.797       817

     accuracy                          0.802      1644
    macro avg      0.802     0.802     0.802      1644
 weighted avg      0.802     0.802     0.802      1644


========== FacebookAI/xlm-roberta-base | task2 ==========


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (11 outputs left out)

epoch 1 | train loss 1.0524 | dev macro-F1 0.3658  <- best
epoch 2 | train loss 0.9342 | dev macro-F1 0.4686  <- best
epoch 3 | train loss 0.8045 | dev macro-F1 0.5129  <- best
epoch 4 | train loss 0.7255 | dev macro-F1 0.5307  <- best

>>> google-bert/bert-base-multilingual-cased | task2 | thresholds [0.3, 0.65, 0.8, 0.75, 0.3]
>>> dev macro-F1 0.5705 | TEST macro-F1 0.5814
               precision    recall  f1-score   support

    political      0.637     0.881     0.739       388
racial_ethnic      0.496     0.480     0.488       125
    religious      0.622     0.431     0.509        65
gender_sexual      0.646     0.520     0.576        98
        other      0.461     0.836     0.595       250

    micro avg      0.559     0.745     0.639       926
    macro avg      0.572     0.630     0.581       926
 weighted avg      0.570     0.745     0.633       926
  samples avg      0.611     0.780     0.656       926


========== google-bert/bert-base-multilingual-cased | task3 ========

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 1 | train loss 0.9057 | dev macro-F1 0.4403  <- best
epoch 2 | train loss 0.8488 | dev macro-F1 0.5032  <- best
epoch 3 | train loss 0.7959 | dev macro-F1 0.4729
epoch 4 | train loss 0.7402 | dev macro-F1 0.4906

>>> google-bert/bert-base-multilingual-cased | task3 | thresholds [0.5, 0.05, 0.45, 0.4, 0.45, 0.5]
>>> dev macro-F1 0.5426 | TEST macro-F1 0.5264
                  precision    recall  f1-score   support

      stereotype      0.320     0.766     0.452       184
    vilification      0.639     1.000     0.780       453
  dehumanization      0.339     0.648     0.445       128
extreme_language      0.412     0.786     0.541       248
 lack_of_empathy      0.314     0.683     0.431       161
    invalidation      0.357     0.896     0.510       231

       micro avg      0.425     0.846     0.566      1405
       macro avg      0.397     0.797     0.526      1405
    weighted avg      0.446     0.846     0.580      1405
     samples avg      0.446     0.850     0.544     

## Step 7. Results table
Saved in `results/pretrained_test_results.csv` (Kaggle: **Output** tab). Download it and put it in the local
`myPolar/results` folder next to the ML and DL results for the report.

In [10]:
os.makedirs("results", exist_ok=True)
res = pd.DataFrame(results)
res.to_csv("results/pretrained_test_results.csv", index=False)

print("TEST macro-F1 (higher is better):")
print(res.pivot_table(index="model", columns="task", values="test_macro_f1").round(3))

TEST macro-F1 (higher is better):
task                          task1  task2  task3
model                                            
bert-base-multilingual-cased  0.741  0.581  0.526
xlm-roberta-base              0.802  0.677  0.479


## Results and findings

**Test macro-F1 (thresholds chosen on dev):**

| Model | Task 1 | Task 2 | Task 3 |
|---|---|---|---|
| XLM-RoBERTa base | **0.802** | **0.677** | 0.479 |
| mBERT (bert-base-multilingual-cased) | 0.741 | 0.581 | **0.526** |

- XLM-RoBERTa gave the best Task 1 and Task 2 scores of the whole project, but only about 0.02 above the one-minute linear baselines.
- XLM-RoBERTa did not learn Task 3: training loss stayed around 0.92 and it predicted almost every label (recall close to 1.0). A lower learning rate, more epochs and several seeds are the next things to try.
- XLM-R splits a Burmese sentence into about 30 pieces, mBERT into about 39 (almost one per character), which may explain part of the gap on Tasks 1 and 2.